# LLM is a great rule-based feature engineer in few-shot tabular learning
## Overview
This notebook runs training and inference for few-shot tabular learning task over benchmark datasets. GPT-3.5 model is used in this tutorial.

## Overall process
* Prepare datasets
* Extract rules for prediction from training samples with the help of LLM
* Parse rules to the program code and convert data into the binary vector
* Train the linear model to predict the likelihood of each class from the binary vector
* Make inference with ensembling

In [1]:
import os
import copy
import utils
import numpy as np
import pandas as pd
import itertools

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from torch.optim import Adam
from sklearn.model_selection import StratifiedKFold

/home/ddmigrushina/.conda/envs/myenv/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


## Prepare datasets
1. Set dataset and simulation parameters (e.g., # of queries for ensemble, # of training shots, and the random seed)
2. Get data and split it into train/test dataset, given simulation parameters

In [2]:
_NUM_QUERY = 5 # Number of ensembles
_SHOT = 4 # Number of training shots [0, 4, 8, 16, 32, 64]
_SEED = 0 # Seed for fixing randomness [864, 460, 142, 629, 761]
_DATA = 'stars' # ['callcenter', 'postpartum', 'stars', 'bank', 'machine', 'crime', 'reading', 'extrovert']
_API_KEY = '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'

_SHOTS = [0, 4, 8, 16, 32, 64]
_SEEDS = [864, 460, 142, 629, 761]
_DATASETS = ['callcenter', 'postpartum', 'stars', 'bank', 'machine', 'crime', 'reading', 'extrovert']


In [3]:
utils.set_seed(_SEED)
df, X_train, X_test, y_train, y_test, target_attr, label_list, is_cat = utils.get_dataset(_DATA, _SHOT, _SEED)

y_train = y_train.astype(str)
y_test = y_test.astype(str)

X_all = df.drop(target_attr, axis=1)
X_train.head(2)

,Vmag,Plx,e_Plx,B-V,SpType,Amag
348,8.17,4.69,1.66,0.36,F2V,16.525864
912,8.1,3.14,0.6,0.043,A0V,15.584648


## Extract rules for prediction from training samples with the help of LLM
To enable the LLM to extract rules based on a more accurate reasoning path, we guided the problem-solving process to mimic how a person might approach a tabular learning task.   

We divided the problem into two sub-tasks for this purpose:   
1. Understand the task description and the features provided by the data, inferring the causal relationships beforehand.   
2. Use the inferred information and few-shot samples to deduce the prediction rules for each class. This two-step reasoning process prevents the model from identifying spurious correlations in irrelevant columns and assists in focusing on more significant features.   

Our prompt comprises three main components as follows:  
* Task description
* Reasoning instruction
* Response instruction

In [4]:
ask_file_name = './templates/ask_llm.txt'
meta_data_name = f"./data/{_DATA}-metadata.json"
templates, feature_desc = utils.get_prompt_for_asking(
    _DATA, X_all, X_train, y_train, label_list, target_attr, ask_file_name, 
    meta_data_name, is_cat, num_query=_NUM_QUERY
)
print(templates[0])

You are an expert. Given the task description and the list of features and data examples, you are extracting conditions for each answer class to solve the task.

Task: Is this star a giant star? Yes or no?


Features:
- Vmag: apparent visual magnitude of the star (brightness as seen from Earth) (numerical variable)
- Plx: parallax of the star in milliarcseconds (measure of distance) (numerical variable)
- e_Plx: error in parallax measurement (numerical variable)
- B-V: color index of the star (difference between blue and visual magnitudes) (numerical variable)
- SpType: spectral type of the star (e.g., G1V, F3V, B8/B9V, K5V) (categorical variable with categories [G1V, F3V, ..., B9III/IV])
- Amag: absolute magnitude of the star (numerical variable)

Examples:
Vmag is 8.1. Plx is 3.14. e_Plx is 0.6. B-V is 0.043. SpType is A0V. Amag is 15.584648.
Answer: 0
Vmag is 8.17. Plx is 4.69. e_Plx is 1.66. B-V is 0.36. SpType is F2V. Amag is 16.525864.
Answer: 0
Vmag is 6.45. Plx is 7.87. e_Plx i

/home/ddmigrushina/FeatLLM/utils.py:325: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(lambda x: x.sample(frac=1))
/home/ddmigrushina/FeatLLM/utils.py:325: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(lambda x: x.sample(frac=1))
/home/ddmigrushina/FeatLLM/utils.py:325: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of panda

In [5]:
_DIVIDER = "\n\n---DIVIDER---\n\n"
_VERSION = "\n\n---VERSION---\n\n"

rule_file_name = f'./rules/rule-{_DATA}-{_SHOT}-{_SEED}.out'
if os.path.isfile(rule_file_name) == False:
    results = utils.query_gpt(templates, _API_KEY, max_tokens=1500, temperature=0.5)
    with open(rule_file_name, 'w') as f:
        total_rules = _DIVIDER.join(results)
        f.write(total_rules)
else:
    with open(rule_file_name, 'r') as f:
        total_rules_str = f.read().strip()
        results = total_rules_str.split(_DIVIDER)

print(results[0])

### Step 1: Analyze the causal relationship or tendency between each feature and task description

1. **Vmag**: Generally, giant stars are brighter than main-sequence stars when viewed from Earth, so a lower apparent visual magnitude (higher brightness) may indicate a giant star.
  
2. **Plx**: A lower parallax indicates a greater distance from Earth. Since giant stars are often more distant than main-sequence stars, a lower parallax might suggest that a star is a giant.

3. **e_Plx**: The error in parallax measurement does not directly indicate whether a star is giant or not but can affect the reliability of distance estimates.

4. **B-V**: A higher B-V color index typically indicates a cooler star, which can correlate with giant stars, especially in later spectral types.

5. **SpType**: Certain spectral types (e.g., III, IV) are indicative of giant stars, while types like V are indicative of main-sequence stars.

6. **Amag**: A higher absolute magnitude (less negative) generally indi

## Parse rules to the program code and convert data into the binary vector

We utilize the rules generated in the previous stage to transform each sample into a binary vector. These vectors are created for each answer class, indicating whether the sample satisfies the rules associated with that class. However, since the rules generated by the LLM are based on natural language, parsing the text into program code is required for automatic data transformation.  

To address the challenges of parsing noisy text, instead of building complex program code, we leverage the LLM itself. We include the function name, input and output descriptions, and inferred rules in the prompt, then input it into the LLM. The generated code is executed using Python’s exec() function along with the provided function name to perform data conversion.

In [6]:
parsed_rules = utils.parse_rules(results, label_list)

saved_file_name = f'./rules/function-{_DATA}-{_SHOT}-{_SEED}.out'    
if os.path.isfile(saved_file_name) == False:
    function_file_name = './templates/ask_for_function.txt'
    fct_strs_all = []
    for parsed_rule in tqdm(parsed_rules):
        fct_templates = utils.get_prompt_for_generating_function(
            parsed_rule, feature_desc, function_file_name
        )
        fct_results = utils.query_gpt(fct_templates, _API_KEY, max_tokens=1500, temperature=0)
        fct_strs = [fct_txt.split('<start>')[1].split('<end>')[0].strip() for fct_txt in fct_results]
        fct_strs_all.append(fct_strs)

    with open(saved_file_name, 'w') as f:
        total_str = _VERSION.join([_DIVIDER.join(x) for x in fct_strs_all])
        f.write(total_str)
else:
    with open(saved_file_name, 'r') as f:
        total_str = f.read().strip()
        fct_strs_all = [x.split(_DIVIDER) for x in total_str.split(_VERSION)]

In [7]:
# Get function names and strings
fct_names = []
fct_strs_final = []
for fct_str_pair in fct_strs_all:
    fct_pair_name = []
    if 'def' not in fct_str_pair[0]:
        continue

    for fct_str in fct_str_pair:
        fct_pair_name.append(fct_str.split('def')[1].split('(')[0].strip())
    fct_names.append(fct_pair_name)
    fct_strs_final.append(fct_str_pair)

In [8]:
print(fct_strs_final[0][0])

def extracting_features_0(df_input):
    df_output = pd.DataFrame()
    df_output['Vmag'] = ((df_input['Vmag'] > 7.0).astype(int) |
                         ((df_input['Vmag'] >= 6.5) & (df_input['Vmag'] < 8.0)).astype(int))
    df_output['Plx'] = ((df_input['Plx'] >= 5.0).astype(int) |
                        ((df_input['Plx'] >= 5.0) & (df_input['Plx'] <= 10.0)).astype(int))
    df_output['B-V'] = ((df_input['B-V'] < 0.5).astype(int) |
                        (df_input['B-V'] <= 0.7).astype(int))
    df_output['SpType'] = (df_input['SpType'].isin(['G1V', 'F3V', 'K5V', 'A0V', 'F2V']).astype(int) |
                           df_input['SpType'].isin(['G2V', 'K1V', 'M0V']).astype(int))
    df_output['Amag'] = ((df_input['Amag'] >= 15.0).astype(int) |
                         (df_input['Amag'] > 14.0).astype(int))
    return df_output


### Convert to binary vectors

In [7]:
label_list = [str(label) for label in label_list]
executable_list, X_train_all_dict, X_test_all_dict = utils.convert_to_binary_vectors(fct_strs_final, fct_names, label_list, X_train, X_test)

NameError: name 'fct_strs_final' is not defined

## Train the linear model to predict the likelihood of each class from the binary vector
When given the rules for each class and a sample, a simple method to measure the class likelihood of the sample is to count how many rules of each class it satisfies (i.e., the sum of the binary vector per class). However, not all rules carry the same importance, necessitating learning their significance from training samples.    
  
We aimed to train this importance using a basic linear model without bias, applied to each class's binary vector.

In [8]:
class simple_model(nn.Module):
    def __init__(self, X):
        super(simple_model, self).__init__()
        self.weights = nn.ParameterList([nn.Parameter(torch.ones(x_each.shape[1] , 1) / x_each.shape[1]) for x_each in X])
        
    def forward(self, x):
        x_total_score = []
        for idx, x_each in enumerate(x):
            x_score = x_each @ torch.clamp(self.weights[idx], min=0)
            x_total_score.append(x_score)
        x_total_score = torch.cat(x_total_score, dim=-1)
        return x_total_score

In [9]:
def train(X_train_now, label_list, shot, y_train_num, y_test_num):
    
    criterion = nn.CrossEntropyLoss()                
    if shot // len(label_list) == 1:
        model = simple_model(X_train_now)
        opt = Adam(model.parameters(), lr=1e-2)
        for _ in range(200):                    
            opt.zero_grad()
            outputs = model(X_train_now)
            preds = outputs.argmax(dim=1).numpy()
            acc = (np.array(y_train_num) == preds).sum() / len(preds)
            if acc == 1:
                break
            loss = criterion(outputs, torch.tensor(y_train_num))
            loss.backward()
            opt.step()
    else:
        if shot // len(label_list) <= 2:
            n_splits = 2
        else:
            n_splits = 4

        kfold = StratifiedKFold(n_splits=n_splits, shuffle=True)
        model_list = []
        for fold, (train_ids, valid_ids) in enumerate(kfold.split(X_train_now[0], y_train_num)):
            model = simple_model(X_train_now)
            opt = Adam(model.parameters(), lr=1e-2)
            X_train_now_fold = [x_train_now[train_ids] for x_train_now in X_train_now]
            X_valid_now_fold = [x_train_now[valid_ids] for x_train_now in X_train_now]
            y_train_fold = y_train_num[train_ids]
            y_valid_fold = y_train_num[valid_ids]

            max_acc = -1
            for _ in range(200):                    
                opt.zero_grad()
                outputs = model(X_train_now_fold)
                loss = criterion(outputs, torch.tensor(y_train_fold))
                loss.backward()
                opt.step()

                valid_outputs = model(X_valid_now_fold)
                preds = valid_outputs.argmax(dim=1).numpy()
                acc = (np.array(y_valid_fold) == preds).sum() / len(preds)
                if max_acc < acc:
                    max_acc = acc 
                    final_model = copy.deepcopy(model)
                    if max_acc >= 1:
                        break
            model_list.append(final_model)

        sdict = model_list[0].state_dict()
        for key in sdict:
            sdict[key] = torch.stack([model.state_dict()[key] for model in model_list], dim=0).mean(dim=0)

        model = simple_model(X_train_now)
        model.load_state_dict(sdict)
    return model

In [ ]:
def get_result(_DATA, _SHOT, _SEED):

    df, X_train, X_test, y_train, y_test, target_attr, label_list, is_cat = utils.get_dataset_TabLLMBench(_DATA, _SHOT, _SEED, use_custom_split=True, ratio='50/50')

    y_train = np.array([str(k) for k in y_train])
    y_test = np.array([str(k) for k in y_test])
    label_list = [str(int(label)) for label in label_list]

    ask_file_name = './templates/ask_llm.txt'
    meta_data_name = f"./data/{_DATA}-metadata.json"
    
    X_all = df.drop(columns=[target_attr])
    
    templates, feature_desc = utils.get_prompt_for_asking(
        _DATA, X_all, X_train, y_train, label_list, target_attr, ask_file_name, 
        meta_data_name, is_cat, num_query=10
    )
    print(templates[0])

    _DIVIDER = "\n\n---DIVIDER---\n\n"
    _VERSION = "\n\n---VERSION---\n\n"

    rule_file_name = f'./rules/rule-{_DATA}-{_SHOT}-{_SEED}.out'
    if os.path.isfile(rule_file_name) == False:
        results = utils.query_gpt(templates, _API_KEY, max_tokens=1500, temperature=0.5)
        with open(rule_file_name, 'w') as f:
            total_rules = _DIVIDER.join(results)
            f.write(total_rules)
    else:
        with open(rule_file_name, 'r') as f:
            total_rules_str = f.read().strip()
            results = total_rules_str.split(_DIVIDER)

    print(results[0])

    parsed_rules = utils.parse_rules(results, label_list)

    saved_file_name = f'./rules/function-{_DATA}-{_SHOT}-{_SEED}.out'    
    if os.path.isfile(saved_file_name) == False:
        function_file_name = './templates/ask_for_function.txt'
        fct_strs_all = []
        for parsed_rule in tqdm(parsed_rules):
            fct_templates = utils.get_prompt_for_generating_function(
                parsed_rule, feature_desc, function_file_name
            )
            fct_results = utils.query_gpt(fct_templates, _API_KEY, max_tokens=1500, temperature=0)
            fct_strs = [fct_txt.split('<start>')[1].split('<end>')[0].strip() for fct_txt in fct_results]
            fct_strs_all.append(fct_strs)

        with open(saved_file_name, 'w') as f:
            total_str = _VERSION.join([_DIVIDER.join(x) for x in fct_strs_all])
            f.write(total_str)
    else:
        with open(saved_file_name, 'r') as f:
            total_str = f.read().strip()
            if total_str:
                fct_strs_all = [x.split(_DIVIDER) for x in total_str.split(_VERSION)]
            else:
                fct_strs_all = []

    fct_names = []
    fct_strs_final = []
    for fct_str_pair in fct_strs_all:
        fct_pair_name = []
        if not fct_str_pair or len(fct_str_pair) == 0:
            continue
        if 'def' not in fct_str_pair[0]:
            continue

        for fct_str in fct_str_pair:
            fct_pair_name.append(fct_str.split('def')[1].split('(')[0].strip())
        fct_names.append(fct_pair_name)
        fct_strs_final.append(fct_str_pair)
    
    executable_list, X_train_all_dict, X_test_all_dict = utils.convert_to_binary_vectors(fct_strs_final, fct_names, label_list, X_train, X_test)
    test_outputs_all = []
    multiclass = True if len(label_list) > 2 else False
    y_train_num = np.array([label_list.index(str(k)) for k in y_train])
    y_test_num = np.array([label_list.index(str(k)) for k in y_test])
    result_auc_all = []
    print(executable_list)
    


    if _SHOT > 0:
        for i in executable_list:
            X_train_now = list(X_train_all_dict[i].values())
            X_test_now = list(X_test_all_dict[i].values())

            trained_model = train(X_train_now, label_list, _SHOT, y_train_num, y_test_num)

            test_outputs = trained_model(X_test_now).detach().cpu()
            test_outputs = F.softmax(test_outputs, dim=1).detach()
            result_auc = utils.evaluate(test_outputs.numpy(), y_test_num, multiclass=multiclass)

            if result_auc < 0.5:
                result_auc = 1 - result_auc
                test_outputs = 1 - test_outputs
                
            print(f"AUC ({_SHOT}-shot):", result_auc)
            test_outputs_all.append(test_outputs)
            result_auc_all.append(result_auc)
    else:
        for i in executable_list:
            X_train_now = list(X_train_all_dict[i].values())
            X_test_now = list(X_test_all_dict[i].values())
        
            model = simple_model(X_test_now)

            with torch.no_grad():
                test_outputs = model(X_test_now).detach().cpu()
                test_outputs = F.softmax(test_outputs, dim=1).detach()

            result_auc = utils.evaluate(test_outputs.numpy(), y_test_num, multiclass=multiclass)

            if result_auc < 0.5:
                result_auc = 1 - result_auc
                test_outputs = 1 - test_outputs
            
            print("AUC (0-shot):", result_auc)
            test_outputs_all.append(test_outputs)
            result_auc_all.append(result_auc)

    
    test_outputs_all = np.stack(test_outputs_all, axis=0)
    ensembled_probs = test_outputs_all.mean(0)
    result_auc = utils.evaluate(ensembled_probs, y_test_num, multiclass=multiclass)
    result_auc_std = np.std(result_auc_all)
    print("Ensembled AUC:", result_auc)
    print("AUC std:", result_auc_std)

    return result_auc, result_auc_std

In [13]:
get_result('extrovert', 4, 864)

You are an expert. Given the task description and the list of features and data examples, you are extracting conditions for each answer class to solve the task.

Task: Does this person maintain long-distance friendships through calls? Yes or no?


Features:
- social_connection_frequency: how often the person connects with others socially (categorical variable with categories [Daily, Weekly, Never, Monthly, weekly])
- gathering_size_preference: preferred size of social gatherings (small gatherings, both, etc.) (categorical variable with categories [Small gatherings, Both, Large events, Never])
- valued_friendship_traits: most valued traits in friendships (support, loyalty, honesty, etc.) (categorical variable with categories [Support, Loyalty, Honesty, All, All the three and trust , Understanding person])
- conflict_resolution_style: preferred approach to resolving conflicts with others (categorical variable with categories [Talk it out, Apologize first, Avoid conflict, Let time fix it]

/home/ddmigrushina/FeatLLM/utils.py:325: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(lambda x: x.sample(frac=1))
/home/ddmigrushina/FeatLLM/utils.py:325: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(lambda x: x.sample(frac=1))
/home/ddmigrushina/FeatLLM/utils.py:325: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of panda

ValueError: need at least one array to stack

In [14]:
#DATASETS = ['bank_credit_scoring', 'callcenter', 'postpartum', 'machine', 'stars', 'crimes_arrest', 'reading', 'extrovert']
DATASETS = ['reading', 'extrovert']
SHOTS = [0, 4, 8, 16, 32, 64]
SEEDS = [864, 460, 142, 629, 761]

def run_all_experiments():
    for data_name, shot, seed in itertools.product(DATASETS, SHOTS, SEEDS):
        print(f"\nRunning: {data_name}, shot={shot}, seed={seed}")
        
        result_file = f'./results/{data_name}_shot{shot}_seed{seed}.csv'
        
        if os.path.isfile(result_file):
            print(f"Already exists: {result_file}")
            continue
        
        try:
            auc, std = get_result(data_name, shot, seed)
            
            df_result = pd.DataFrame([{
                'dataset': data_name,
                'shot': shot,
                'seed': seed,
                'auc': auc,
                'std': std
            }])
            
            os.makedirs('./results', exist_ok=True)
            df_result.to_csv(result_file, index=False)
            print(f"Saved: {result_file}")
            
        except Exception as e:
            print(f"Error on {data_name}, shot={shot}, seed={seed}: {e}")
            df_error = pd.DataFrame([{
                'dataset': data_name,
                'shot': shot,
                'seed': seed,
                'auc': None,
                'std': None,
                'error': str(e)
            }])
            df_error.to_csv(result_file, index=False)

def aggregate_results():
    summary_data = []

    DATASETS = ['bank_credit_scoring', 'callcenter', 'postpartum', 'machine', 'stars', 'crimes_arrest', 'reading', 'extrovert']
    SHOTS = [0, 4, 8, 16, 32, 64]
    
    for data_name, shot in itertools.product(DATASETS, SHOTS):
        aucs = []
        stds = []
        
        for seed in SEEDS:
            result_file = f'./results/{data_name}_shot{shot}_seed{seed}.csv'
            
            if os.path.isfile(result_file):
                df = pd.read_csv(result_file)
                if 'auc' in df.columns and pd.notna(df['auc'].iloc[0]):
                    aucs.append(df['auc'].iloc[0])
                    if 'std' in df.columns:
                        stds.append(df['std'].iloc[0])
        
        if aucs:
            summary_data.append({
                'dataset': data_name,
                'shot': shot,
                'mean_auc': np.mean(aucs),
                'std_auc': np.std(aucs, ddof=1),
                'mean_std': np.mean(stds) if stds else 0,
                'n_success': len(aucs)
            })
    
    df_summary = pd.DataFrame(summary_data)
    df_summary.to_csv('./results/summary.csv', index=False)
    print("Summary saved to ./results/summary.csv")
    return df_summary

In [15]:
run_all_experiments()
aggregate_results()



Running: reading, shot=0, seed=864
Error on reading, shot=0, seed=864: sequence item 4: expected str instance, float found

Running: reading, shot=0, seed=460
Error on reading, shot=0, seed=460: sequence item 4: expected str instance, float found

Running: reading, shot=0, seed=142
Error on reading, shot=0, seed=142: sequence item 4: expected str instance, float found

Running: reading, shot=0, seed=629
Error on reading, shot=0, seed=629: sequence item 4: expected str instance, float found

Running: reading, shot=0, seed=761
Error on reading, shot=0, seed=761: sequence item 4: expected str instance, float found

Running: reading, shot=4, seed=864
Error on reading, shot=4, seed=864: sequence item 4: expected str instance, float found

Running: reading, shot=4, seed=460
Error on reading, shot=4, seed=460: sequence item 4: expected str instance, float found

Running: reading, shot=4, seed=142
Error on reading, shot=4, seed=142: sequence item 4: expected str instance, float found

Running

/home/ddmigrushina/FeatLLM/utils.py:285: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(lambda x: x.sample(sample_num))
/home/ddmigrushina/FeatLLM/utils.py:285: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(lambda x: x.sample(sample_num))
/home/ddmigrushina/FeatLLM/utils.py:285: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version 

Error on reading, shot=64, seed=460: sequence item 4: expected str instance, float found

Running: reading, shot=64, seed=142
Error on reading, shot=64, seed=142: sequence item 4: expected str instance, float found

Running: reading, shot=64, seed=629
Error on reading, shot=64, seed=629: sequence item 4: expected str instance, float found

Running: reading, shot=64, seed=761
Error on reading, shot=64, seed=761: sequence item 4: expected str instance, float found

Running: extrovert, shot=0, seed=864
You are an expert. Given the task description and the list of features and data examples, you are extracting conditions for each answer class to solve the task.

Task: Does this person maintain long-distance friendships through calls? Yes or no?


Features:
- social_connection_frequency: how often the person connects with others socially (categorical variable with categories [Daily, Weekly, Never, Monthly, weekly])
- gathering_size_preference: preferred size of social gatherings (small gath

100%|██████████| 5/5 [01:03<00:00, 12.66s/it]


Step 1. The relationship between each feature and the task description:

- **social_connection_frequency**: Higher frequency (Daily, Weekly) suggests active maintenance of friendships, including long-distance ones through calls. Lower frequency (Never, Monthly) suggests less likelihood.
- **gathering_size_preference**: Preference for small gatherings or both may indicate valuing close connections, possibly maintaining long-distance friendships. Preference for large events or never may indicate less intimate or less frequent contact.
- **valued_friendship_traits**: Valuing support, loyalty, honesty, or all traits suggests a strong commitment to friendships, likely maintaining long-distance friendships.
- **conflict_resolution_style**: Those who "Talk it out" or "Apologize first" may be more proactive in maintaining relationships, including long-distance ones. Avoiding conflict or letting time fix it may indicate less active maintenance.
- **family_support_level**: Higher family support 

...

Step 1. The relationship between each feature and the task description:

- **social_connection_frequency**: People who maintain long-distance friendships through calls are likely to connect frequently (Daily, Weekly, Monthly) rather than Never.
- **gathering_size_preference**: Preference for small gatherings or both small and large may indicate valuing close personal connections, which may translate to maintaining long-distance friendships.
- **valued_friendship_traits**: Valuing traits like support, loyalty, honesty, and understanding likely correlates with maintaining long-distance friendships through calls to sustain these traits.
- **conflict_resolution_style**: Those who prefer to "Talk it out" or "Apologize first" may maintain friendships actively, including long-distance ones.
- **family_support_level**: Higher family support (Yes, Sometimes) may reflect a socially connected person who also maintains friendships.
- **social_battery_recharge**: People who recharge by "Talking to 

0it [00:00, ?it/s]/4 [00:00<?, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
100%|██████████| 4/4 [00:12<00:00,  3.04s/it]
/home/ddmigrushina/FeatLLM/utils.py:325: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(lambda x: x.sample(frac=1))
/home/ddmigrushina/FeatLLM/utils.py:325: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(lambda x: x.sample(frac=1))
/home/ddmigrushina/FeatLLM/utils.py:325: FutureWarni

[]
Error on extrovert, shot=0, seed=761: need at least one array to stack

Running: extrovert, shot=4, seed=864
You are an expert. Given the task description and the list of features and data examples, you are extracting conditions for each answer class to solve the task.

Task: Does this person maintain long-distance friendships through calls? Yes or no?


Features:
- social_connection_frequency: how often the person connects with others socially (categorical variable with categories [Daily, Weekly, Never, Monthly, weekly])
- gathering_size_preference: preferred size of social gatherings (small gatherings, both, etc.) (categorical variable with categories [Small gatherings, Both, Large events, Never])
- valued_friendship_traits: most valued traits in friendships (support, loyalty, honesty, etc.) (categorical variable with categories [Support, Loyalty, Honesty, All, All the three and trust , Understanding person])
- conflict_resolution_style: preferred approach to resolving conflicts w

 20%|██        | 1/5 [00:20<01:20, 20.13s/it]


KeyboardInterrupt: 